In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Project 18 — Local Customer Support Memory Bot
## Retrieve Similar Tickets and Suggested Fixes

**Stack:** Ollama · LangChain · ChromaDB · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Setup

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.schema import Document

llm = ChatOllama(model="qwen3:8b", temperature=0.2)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## Step 2 — Historical Ticket Database

In [ ]:
tickets = [
    Document(page_content="""Ticket #1042: User cannot login after password reset
Customer: john@example.com | Priority: High | Status: Resolved
Issue: After resetting password, user gets "Invalid credentials" error.
Root Cause: Password reset token expired (15-min window).
Resolution: Generated new reset link, extended token expiry to 30 min.
Time to resolve: 2 hours""",
        metadata={"ticket_id": 1042, "category": "auth", "resolved": True}),

    Document(page_content="""Ticket #1055: Dashboard loading slowly
Customer: corp@bigco.com | Priority: Medium | Status: Resolved
Issue: Dashboard takes 30+ seconds to load for accounts with 10K+ records.
Root Cause: Missing database index on the analytics table.
Resolution: Added composite index on (account_id, created_at). Load time: <2s.
Time to resolve: 4 hours""",
        metadata={"ticket_id": 1055, "category": "performance", "resolved": True}),

    Document(page_content="""Ticket #1063: API rate limit exceeded
Customer: dev@startup.io | Priority: Medium | Status: Resolved
Issue: Getting 429 errors during automated data imports.
Root Cause: Customer hitting 100 req/min limit with bulk import script.
Resolution: Provided batch import endpoint (/api/v2/bulk-import).
Time to resolve: 1 hour""",
        metadata={"ticket_id": 1063, "category": "api", "resolved": True}),

    Document(page_content="""Ticket #1078: Email notifications not sending
Customer: admin@school.edu | Priority: High | Status: Resolved
Issue: No email notifications for new comments in the last 24 hours.
Root Cause: SMTP relay service (SendGrid) had regional outage.
Resolution: Switched to backup SMTP provider. Queued emails sent.
Time to resolve: 6 hours""",
        metadata={"ticket_id": 1078, "category": "notifications", "resolved": True}),

    Document(page_content="""Ticket #1091: Data export missing columns
Customer: analyst@finance.com | Priority: Low | Status: Resolved
Issue: CSV export missing 'created_at' and 'updated_at' columns.
Root Cause: Export template not updated after schema migration.
Resolution: Updated export template, added regression test.
Time to resolve: 3 hours""",
        metadata={"ticket_id": 1091, "category": "data", "resolved": True}),
]
print(f"Historical ticket database: {len(tickets)} resolved tickets")

## Step 3 — Build Ticket Memory

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(tickets, embeddings,
    persist_directory="sample_data/support_chroma", collection_name="tickets")
print("Support ticket memory indexed!")

## Step 4 — Similar Ticket Finder + Resolution Suggester

In [ ]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

support_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a support agent assistant. A new ticket has come in.
Use the similar historical tickets to:
1. Identify the most likely root cause
2. Suggest a resolution based on past fixes
3. Estimate time to resolve
4. Recommend which team should handle it

Similar past tickets:
{context}

New ticket description: {question}

Analysis and recommended resolution:"""
)

support_qa = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": support_prompt},
)

new_tickets = [
    "User reports they can't login. They say the password reset email link is expired.",
    "Customer's API calls are getting rejected with HTTP 429 errors.",
    "The weekly report email didn't arrive for any users this morning.",
]

for ticket in new_tickets:
    print(f"\n{'='*60}")
    print(f"NEW TICKET: {ticket}")
    result = support_qa.invoke({"query": ticket})
    print(f"\nSUGGESTED RESPONSE:\n{result['result']}")
    similar = [s.metadata.get('ticket_id') for s in result['source_documents']]
    print(f"\nSimilar tickets: {similar}")

## What You Learned
- **Ticket knowledge base** with resolution history
- **Similar ticket retrieval** for faster resolution
- **Resolution suggestion** from historical patterns